# Seed-42 qualitative evaluation figures

Requires final checkpoint and dataset. Source preserved; verify saved no-TTA threshold.

Source: `graindiscoloration (2).ipynb`, zero-based cell 62.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
# ================================================================
# ONE-CELL PAPER-READY EVALUATION FIGURE GENERATOR
# Final six-channel proposed model, seed 42, no TTA
# ================================================================

from pathlib import Path
import zipfile
import shutil
import re
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ------------------------------------------------
# 1–2. SETTINGS AND AUTOMATIC DATASET/FOLDER DETECTION
# Replace the previous ZIP-search section with this
# ------------------------------------------------

from pathlib import Path
import zipfile
import shutil
import re

VALIDATION_SELECTED_THRESHOLD = 0.60
NUMBER_OF_THRESHOLD_BINS = 1000

WORK_DIR = Path("/kaggle/working")
INPUT_DIR = Path("/kaggle/input")

OUTPUT_DIR = WORK_DIR / "gd_seed42_paper_ready_figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def normalize_source_name(text):
    """Remove hyphens, underscores and other symbols."""
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(text).lower(),
    )


TARGET_NORMALIZED_NAME = normalize_source_name(
    "gd_v3_1_without_ndre_seed42_results"
)

# Find matching Kaggle input dataset folder
dataset_candidates = []

if INPUT_DIR.exists():
    for path in INPUT_DIR.iterdir():
        if not path.is_dir():
            continue

        normalized_path_name = normalize_source_name(path.name)

        if (
            TARGET_NORMALIZED_NAME in normalized_path_name
            or normalized_path_name in TARGET_NORMALIZED_NAME
        ):
            dataset_candidates.append(path)

# Also search for matching folders one level deeper
if not dataset_candidates and INPUT_DIR.exists():
    for path in INPUT_DIR.glob("*/*"):
        if not path.is_dir():
            continue

        normalized_path_name = normalize_source_name(path.name)

        if TARGET_NORMALIZED_NAME in normalized_path_name:
            dataset_candidates.append(path)

if not dataset_candidates:
    print("Available Kaggle input folders:")

    if INPUT_DIR.exists():
        for path in INPUT_DIR.iterdir():
            if path.is_dir():
                print(" •", path)

    raise FileNotFoundError(
        "\nCould not locate the Kaggle dataset folder.\n"
        "Wait until 'Adding data source' finishes, "
        "then run the cell again."
    )

# Prefer the shortest/root-level matching folder
dataset_candidates.sort(key=lambda path: len(str(path)))
source_dataset_dir = dataset_candidates[0]

print("Dataset folder found:")
print(source_dataset_dir)

# Check whether the dataset itself contains a ZIP
internal_zip_candidates = [
    path for path in source_dataset_dir.rglob("*.zip")
    if normalize_source_name(
        path.stem
    ).startswith("gdv31withoutndreseed42")
]

if internal_zip_candidates:
    # Dataset contains a ZIP file
    result_zip_path = internal_zip_candidates[0]

    EXTRACT_DIR = OUTPUT_DIR / "extracted_results"
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(result_zip_path, "r") as archive:
        archive.extractall(EXTRACT_DIR)

    print("\nInternal ZIP found:")
    print(result_zip_path)
    print("Extracted to:")
    print(EXTRACT_DIR)

else:
    # Kaggle already extracted the files
    EXTRACT_DIR = source_dataset_dir

    # Keep this variable because the remaining code references it
    result_zip_path = source_dataset_dir

    print("\nFiles are already extracted by Kaggle.")
    print("Reading directly from:")
    print(EXTRACT_DIR)

# Verify that training history exists
history_check = list(
    EXTRACT_DIR.rglob("training_history.csv")
)

if not history_check:
    print("\nFiles currently visible:")

    visible_files = [
        path for path in EXTRACT_DIR.rglob("*")
        if path.is_file()
    ]

    for path in visible_files[:100]:
        print(" •", path.relative_to(EXTRACT_DIR))

    raise FileNotFoundError(
        "\nDataset folder was found, but "
        "training_history.csv is not visible.\n"
        "The Kaggle data source may still be loading."
    )

print("\nTraining history found:")
print(history_check[0])

extracted_files = [
    path for path in EXTRACT_DIR.rglob("*")
    if path.is_file()
]

print("\nAvailable result files:")
for path in extracted_files:
    print(" •", path.relative_to(EXTRACT_DIR))

print("\nSource detection completed successfully.")

# ------------------------------------------------
# 3. HELPER FUNCTIONS
# ------------------------------------------------

def canonical_name(text):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(text).strip().lower(),
    ).strip("_")


def find_column(dataframe, possible_names):
    column_lookup = {
        canonical_name(column): column
        for column in dataframe.columns
    }

    for possible_name in possible_names:
        key = canonical_name(possible_name)

        if key in column_lookup:
            return column_lookup[key]

    return None


def find_global_object(names):
    for name in names:
        if name in globals() and globals()[name] is not None:
            return globals()[name], name

    return None, None


def extract_logits(output):
    import torch

    if torch.is_tensor(output):
        return output

    if isinstance(output, (tuple, list)):
        if len(output) > 0 and torch.is_tensor(output[0]):
            return output[0]

    if isinstance(output, dict):
        for key in ("logits", "out", "prediction", "pred"):
            if key in output and torch.is_tensor(output[key]):
                return output[key]

    raise TypeError("Could not identify final segmentation logits.")


def inference_probabilities(model, loader, device):
    """
    No-TTA inference. Used only if saved probabilities
    are not found inside the result ZIP.
    """

    import torch
    import torch.nn.functional as F

    model = model.to(device)
    model.eval()

    all_targets = []
    all_probabilities = []

    with torch.inference_mode():
        for batch in loader:
            if not isinstance(batch, (tuple, list)) or len(batch) < 2:
                raise RuntimeError(
                    "Loader must return at least images and masks."
                )

            images, masks = batch[0], batch[1]

            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            if masks.ndim == 3:
                masks = masks.unsqueeze(1)

            logits = extract_logits(model(images))

            if logits.ndim == 3:
                logits = logits.unsqueeze(1)

            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(
                    logits,
                    size=masks.shape[-2:],
                    mode="bilinear",
                    align_corners=False,
                )

            if logits.shape[1] == 1:
                probabilities = torch.sigmoid(logits)

            elif logits.shape[1] == 2:
                probabilities = torch.softmax(
                    logits,
                    dim=1,
                )[:, 1:2]

            else:
                raise ValueError(
                    f"Expected 1 or 2 output channels, "
                    f"got {logits.shape[1]}."
                )

            targets = masks >= 0.5

            all_targets.append(
                targets.detach()
                .cpu()
                .numpy()
                .astype(np.uint8)
                .ravel()
            )

            all_probabilities.append(
                probabilities.detach()
                .cpu()
                .numpy()
                .astype(np.float32)
                .ravel()
            )

    if not all_targets:
        raise RuntimeError("No samples were processed.")

    return (
        np.concatenate(all_targets),
        np.concatenate(all_probabilities),
    )


def identify_array(npz_data, possible_keys):
    key_lookup = {
        canonical_name(key): key
        for key in npz_data.files
    }

    for possible_key in possible_keys:
        key = canonical_name(possible_key)

        if key in key_lookup:
            return npz_data[key_lookup[key]]

    return None


def load_saved_probabilities(split_name):
    """
    Searches for saved validation/test pixel probabilities.

    Supported:
    1. NPZ containing targets and probabilities
    2. Separate .npy probability and target files
    3. Current notebook model + loader fallback
    """

    split_name = split_name.lower()

    target_keys = [
        "y_true",
        "targets",
        "target",
        "masks",
        "mask",
        "ground_truth",
        "ground_truths",
        "gt",
        "labels",
    ]

    probability_keys = [
        "y_prob",
        "probabilities",
        "probability",
        "probs",
        "pred_probs",
        "prediction_probabilities",
        "soft_predictions",
    ]

    # ---------- NPZ search ----------
    npz_files = list(EXTRACT_DIR.rglob("*.npz"))

    npz_files.sort(
        key=lambda path: (
            0 if split_name in path.name.lower() else 1,
            0 if "prob" in path.name.lower() else 1,
        )
    )

    for path in npz_files:
        name = path.name.lower()

        if (
            "val" not in name
            and "valid" not in name
            and "test" not in name
        ):
            # Still inspect if only one generic prediction file exists
            pass
        elif split_name == "validation":
            if "val" not in name and "valid" not in name:
                continue
        elif split_name == "test":
            if "test" not in name:
                continue

        try:
            data = np.load(path, allow_pickle=True)

            targets = identify_array(data, target_keys)
            probabilities = identify_array(data, probability_keys)

            if targets is None or probabilities is None:
                continue

            targets = np.asarray(targets).ravel()
            probabilities = np.asarray(
                probabilities,
                dtype=np.float32,
            ).ravel()

            if len(targets) != len(probabilities):
                continue

            # Convert logits if necessary
            finite_probabilities = probabilities[
                np.isfinite(probabilities)
            ]

            if len(finite_probabilities) == 0:
                continue

            if (
                finite_probabilities.min() < 0
                or finite_probabilities.max() > 1
            ):
                probabilities = (
                    1.0 / (1.0 + np.exp(-probabilities))
                ).astype(np.float32)

            targets = (targets >= 0.5).astype(np.uint8)

            if np.unique(probabilities).size <= 2:
                print(
                    f"Skipped binary predictions without "
                    f"probabilities: {path.name}"
                )
                continue

            print(
                f"{split_name.title()} probabilities loaded from:",
                path,
            )

            return targets, probabilities

        except Exception:
            continue

    # ---------- Separate NPY file search ----------
    all_npy = list(EXTRACT_DIR.rglob("*.npy"))

    probability_files = [
        path for path in all_npy
        if split_name[:4] in path.name.lower()
        and any(
            word in path.name.lower()
            for word in ("prob", "score", "soft")
        )
    ]

    target_files = [
        path for path in all_npy
        if split_name[:4] in path.name.lower()
        and any(
            word in path.name.lower()
            for word in ("target", "mask", "ground", "label", "gt")
        )
    ]

    for probability_path in probability_files:
        for target_path in target_files:
            try:
                probabilities = np.load(
                    probability_path,
                    allow_pickle=True,
                ).astype(np.float32).ravel()

                targets = np.load(
                    target_path,
                    allow_pickle=True,
                ).ravel()

                if len(targets) != len(probabilities):
                    continue

                if probabilities.min() < 0 or probabilities.max() > 1:
                    probabilities = (
                        1.0 / (1.0 + np.exp(-probabilities))
                    ).astype(np.float32)

                targets = (targets >= 0.5).astype(np.uint8)

                if np.unique(probabilities).size <= 2:
                    continue

                print(
                    f"{split_name.title()} probabilities loaded from:",
                    probability_path,
                )
                print("Targets loaded from:", target_path)

                return targets, probabilities

            except Exception:
                continue

    # ---------- Current-session inference fallback ----------
    current_model, model_name = find_global_object([
        "model",
        "proposed_model",
        "final_model",
    ])

    if split_name == "validation":
        current_loader, loader_name = find_global_object([
            "val_loader",
            "valid_loader",
            "validation_loader",
        ])
    else:
        current_loader, loader_name = find_global_object([
            "test_loader",
            "testing_loader",
        ])

    current_device, device_name = find_global_object(["device"])

    if (
        current_model is not None
        and current_loader is not None
    ):
        if current_device is None:
            import torch

            current_device = torch.device(
                "cuda" if torch.cuda.is_available()
                else "cpu"
            )

        print(
            f"No saved {split_name} probability array found."
        )
        print(
            f"Running no-TTA inference using "
            f"{model_name} and {loader_name}."
        )

        return inference_probabilities(
            current_model,
            current_loader,
            current_device,
        )

    return None, None


def threshold_analysis(y_true, y_prob, number_of_bins=1000):
    edges = np.linspace(
        0.0,
        1.0,
        number_of_bins + 1,
    )

    positive_histogram, _ = np.histogram(
        y_prob[y_true == 1],
        bins=edges,
    )

    negative_histogram, _ = np.histogram(
        y_prob[y_true == 0],
        bins=edges,
    )

    thresholds = edges[:-1]

    tp = np.cumsum(
        positive_histogram[::-1]
    )[::-1].astype(np.float64)

    fp = np.cumsum(
        negative_histogram[::-1]
    )[::-1].astype(np.float64)

    total_positive = float(
        positive_histogram.sum()
    )

    fn = total_positive - tp

    precision = np.divide(
        tp,
        tp + fp,
        out=np.zeros_like(tp),
        where=(tp + fp) > 0,
    )

    recall = np.divide(
        tp,
        tp + fn,
        out=np.zeros_like(tp),
        where=(tp + fn) > 0,
    )

    dice = np.divide(
        2.0 * tp,
        2.0 * tp + fp + fn,
        out=np.zeros_like(tp),
        where=(2.0 * tp + fp + fn) > 0,
    )

    iou = np.divide(
        tp,
        tp + fp + fn,
        out=np.zeros_like(tp),
        where=(tp + fp + fn) > 0,
    )

    return pd.DataFrame({
        "threshold": thresholds,
        "dice": dice,
        "iou": iou,
        "precision": precision,
        "recall": recall,
    })


# ------------------------------------------------
# 4. LOAD AND STANDARDISE TRAINING HISTORY
# ------------------------------------------------

history_candidates = list(
    EXTRACT_DIR.rglob("training_history.csv")
)

if not history_candidates:
    history_candidates = [
        path for path in EXTRACT_DIR.rglob("*.csv")
        if "history" in path.name.lower()
    ]

if not history_candidates:
    raise FileNotFoundError(
        "training_history.csv was not found inside the ZIP."
    )

history_path = history_candidates[0]
raw_history = pd.read_csv(history_path)

print("\nTraining history:", history_path)
print("Original columns:", list(raw_history.columns))

epoch_column = find_column(
    raw_history,
    ["epoch", "epochs", "epoch_number"],
)

train_loss_column = find_column(
    raw_history,
    [
        "train_loss",
        "training_loss",
        "epoch_loss",
        "total_train_loss",
        "total_loss",
        "loss",
    ],
)

val_loss_column = find_column(
    raw_history,
    [
        "val_loss",
        "valid_loss",
        "validation_loss",
    ],
)

val_dice_column = find_column(
    raw_history,
    [
        "val_dice",
        "valid_dice",
        "validation_dice",
        "val_dice_score",
    ],
)

val_iou_column = find_column(
    raw_history,
    [
        "val_iou",
        "valid_iou",
        "validation_iou",
        "val_jaccard",
    ],
)

val_precision_column = find_column(
    raw_history,
    [
        "val_precision",
        "valid_precision",
        "validation_precision",
    ],
)

val_recall_column = find_column(
    raw_history,
    [
        "val_recall",
        "valid_recall",
        "validation_recall",
    ],
)

val_threshold_column = find_column(
    raw_history,
    [
        "val_threshold",
        "valid_threshold",
        "validation_threshold",
        "best_threshold",
        "threshold",
    ],
)

lr_column = find_column(
    raw_history,
    [
        "lr",
        "learning_rate",
        "current_lr",
    ],
)

if epoch_column is None:
    raw_history["__epoch__"] = np.arange(
        1,
        len(raw_history) + 1,
    )
    epoch_column = "__epoch__"

if train_loss_column is None:
    raise ValueError(
        "Training-loss column was not identified.\n"
        f"Available columns: {list(raw_history.columns)}"
    )

if val_dice_column is None:
    raise ValueError(
        "Validation-Dice column was not identified.\n"
        f"Available columns: {list(raw_history.columns)}"
    )

history = pd.DataFrame({
    "epoch": pd.to_numeric(
        raw_history[epoch_column],
        errors="coerce",
    ),
    "train_loss": pd.to_numeric(
        raw_history[train_loss_column],
        errors="coerce",
    ),
    "val_dice": pd.to_numeric(
        raw_history[val_dice_column],
        errors="coerce",
    ),
})

if val_loss_column is not None:
    history["val_loss"] = pd.to_numeric(
        raw_history[val_loss_column],
        errors="coerce",
    )

if val_iou_column is not None:
    history["val_iou"] = pd.to_numeric(
        raw_history[val_iou_column],
        errors="coerce",
    )

if val_precision_column is not None:
    history["val_precision"] = pd.to_numeric(
        raw_history[val_precision_column],
        errors="coerce",
    )

if val_recall_column is not None:
    history["val_recall"] = pd.to_numeric(
        raw_history[val_recall_column],
        errors="coerce",
    )

if val_threshold_column is not None:
    history["val_threshold"] = pd.to_numeric(
        raw_history[val_threshold_column],
        errors="coerce",
    )

if lr_column is not None:
    history["learning_rate"] = pd.to_numeric(
        raw_history[lr_column],
        errors="coerce",
    )

history = history.dropna(
    subset=["epoch", "train_loss", "val_dice"]
).sort_values("epoch").reset_index(drop=True)

history.to_csv(
    OUTPUT_DIR / "training_history_standardized.csv",
    index=False,
)

best_index = history["val_dice"].idxmax()
best_epoch = int(history.loc[best_index, "epoch"])
best_val_dice = float(
    history.loc[best_index, "val_dice"]
)

print(
    f"Best logged validation Dice = "
    f"{best_val_dice:.4f} at epoch {best_epoch}"
)

# ------------------------------------------------
# 5. LOAD VALIDATION PROBABILITIES
# ------------------------------------------------

validation_targets, validation_probabilities = (
    load_saved_probabilities("validation")
)

threshold_table = None

if (
    validation_targets is not None
    and validation_probabilities is not None
):
    threshold_table = threshold_analysis(
        validation_targets,
        validation_probabilities,
        NUMBER_OF_THRESHOLD_BINS,
    )

    threshold_table.to_csv(
        OUTPUT_DIR / "validation_threshold_analysis.csv",
        index=False,
    )

# ------------------------------------------------
# 6. CREATE COMBINED CONVERGENCE FIGURE
# ------------------------------------------------

figure = plt.figure(figsize=(10.6, 7.4))

grid = figure.add_gridspec(
    2,
    2,
    height_ratios=[1.0, 1.08],
    hspace=0.38,
    wspace=0.28,
)

axis_loss = figure.add_subplot(grid[0, 0])
axis_validation = figure.add_subplot(grid[0, 1])
axis_bottom = figure.add_subplot(grid[1, :])

# (a) Training objective
axis_loss.plot(
    history["epoch"],
    history["train_loss"],
    color="#2C7FB8",
    linewidth=2.1,
    label="Training objective",
)

# Validation loss is deliberately not overlaid by default.
# Training objective may contain auxiliary loss terms.
axis_loss.set_xlabel("Epoch")
axis_loss.set_ylabel("Training objective")
axis_loss.grid(alpha=0.22, linewidth=0.7)
axis_loss.legend(frameon=False)
axis_loss.text(
    0.02,
    0.96,
    "(a)",
    transform=axis_loss.transAxes,
    va="top",
    fontweight="bold",
)

# (b) Validation metrics
axis_validation.plot(
    history["epoch"],
    history["val_dice"],
    color="#D95F02",
    linewidth=2.1,
    label="Validation Dice",
)

if "val_iou" in history.columns:
    axis_validation.plot(
        history["epoch"],
        history["val_iou"],
        color="#1B9E77",
        linewidth=1.9,
        label="Validation IoU",
    )

axis_validation.axvline(
    best_epoch,
    color="#444444",
    linestyle="--",
    linewidth=1.2,
    label=f"Selected epoch = {best_epoch}",
)

axis_validation.scatter(
    [best_epoch],
    [best_val_dice],
    s=62,
    color="#D95F02",
    edgecolor="white",
    linewidth=0.8,
    zorder=5,
)

axis_validation.set_xlabel("Epoch")
axis_validation.set_ylabel("Validation score")
axis_validation.set_ylim(0.0, 1.02)
axis_validation.grid(alpha=0.22, linewidth=0.7)
axis_validation.legend(
    frameon=False,
    loc="lower right",
)
axis_validation.text(
    0.02,
    0.96,
    "(b)",
    transform=axis_validation.transAxes,
    va="top",
    fontweight="bold",
)

# (c) Threshold sensitivity, or validation P/R fallback
if threshold_table is not None:
    axis_bottom.plot(
        threshold_table["threshold"],
        threshold_table["dice"],
        color="#D95F02",
        linewidth=2.2,
        label="Dice",
    )

    axis_bottom.plot(
        threshold_table["threshold"],
        threshold_table["precision"],
        color="#7570B3",
        linewidth=1.9,
        label="Precision",
    )

    axis_bottom.plot(
        threshold_table["threshold"],
        threshold_table["recall"],
        color="#1B9E77",
        linewidth=1.9,
        label="Recall",
    )

    axis_bottom.axvline(
        VALIDATION_SELECTED_THRESHOLD,
        color="#222222",
        linestyle="--",
        linewidth=1.4,
        label=(
            f"Selected threshold = "
            f"{VALIDATION_SELECTED_THRESHOLD:.2f}"
        ),
    )

    selected_index = np.abs(
        threshold_table["threshold"].to_numpy()
        - VALIDATION_SELECTED_THRESHOLD
    ).argmin()

    selected_row = threshold_table.iloc[selected_index]

    axis_bottom.scatter(
        [VALIDATION_SELECTED_THRESHOLD],
        [selected_row["dice"]],
        s=66,
        color="#D95F02",
        edgecolor="white",
        zorder=5,
    )

    axis_bottom.text(
        min(0.78, VALIDATION_SELECTED_THRESHOLD + 0.025),
        max(0.08, selected_row["dice"] - 0.18),
        (
            f"Dice = {selected_row['dice']:.3f}\n"
            f"Precision = {selected_row['precision']:.3f}\n"
            f"Recall = {selected_row['recall']:.3f}"
        ),
        fontsize=9,
        bbox={
            "facecolor": "white",
            "edgecolor": "#BBBBBB",
            "alpha": 0.92,
        },
    )

    axis_bottom.set_xlabel("Segmentation threshold")
    axis_bottom.set_ylabel("Validation score")
    axis_bottom.set_xlim(0.0, 1.0)
    axis_bottom.set_ylim(0.0, 1.02)
    axis_bottom.legend(
        frameon=False,
        ncol=4,
        loc="lower center",
    )

else:
    plotted_something = False

    if "val_precision" in history.columns:
        axis_bottom.plot(
            history["epoch"],
            history["val_precision"],
            color="#7570B3",
            linewidth=2.0,
            label="Validation Precision",
        )
        plotted_something = True

    if "val_recall" in history.columns:
        axis_bottom.plot(
            history["epoch"],
            history["val_recall"],
            color="#1B9E77",
            linewidth=2.0,
            label="Validation Recall",
        )
        plotted_something = True

    if "val_threshold" in history.columns:
        axis_bottom.plot(
            history["epoch"],
            history["val_threshold"],
            color="#444444",
            linewidth=1.8,
            linestyle="--",
            label="Validation-selected threshold",
        )
        plotted_something = True

    if not plotted_something:
        axis_bottom.text(
            0.5,
            0.5,
            "Validation pixel probabilities were not found.\n"
            "Threshold-sensitivity analysis was not generated.",
            transform=axis_bottom.transAxes,
            ha="center",
            va="center",
        )

    axis_bottom.set_xlabel("Epoch")
    axis_bottom.set_ylabel("Validation value")
    axis_bottom.set_ylim(0.0, 1.02)

    if plotted_something:
        axis_bottom.legend(
            frameon=False,
            ncol=3,
            loc="lower center",
        )

axis_bottom.grid(alpha=0.22, linewidth=0.7)
axis_bottom.text(
    0.01,
    0.96,
    "(c)",
    transform=axis_bottom.transAxes,
    va="top",
    fontweight="bold",
)

convergence_png = (
    OUTPUT_DIR
    / "figure_convergence_and_validation_analysis.png"
)

convergence_pdf = (
    OUTPUT_DIR
    / "figure_convergence_and_validation_analysis.pdf"
)

figure.savefig(
    convergence_png,
    dpi=600,
    bbox_inches="tight",
)

figure.savefig(
    convergence_pdf,
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

print("\nCreated:", convergence_png)
print("Created:", convergence_pdf)

# ------------------------------------------------
# 7. TEST PR CURVE AND AUPRC
# ------------------------------------------------

test_targets, test_probabilities = (
    load_saved_probabilities("test")
)

auprc_value = None

if (
    test_targets is not None
    and test_probabilities is not None
):
    try:
        from sklearn.metrics import (
            precision_recall_curve,
            average_precision_score,
        )

        auprc_value = float(
            average_precision_score(
                test_targets,
                test_probabilities,
            )
        )

        try:
            precision_curve, recall_curve, _ = (
                precision_recall_curve(
                    test_targets,
                    test_probabilities,
                    drop_intermediate=True,
                )
            )
        except TypeError:
            precision_curve, recall_curve, _ = (
                precision_recall_curve(
                    test_targets,
                    test_probabilities,
                )
            )

        # Reduce plotted points only; exact AUPRC is retained
        if len(recall_curve) > 5000:
            indices = np.linspace(
                0,
                len(recall_curve) - 1,
                5000,
            ).astype(int)

            plotted_recall = recall_curve[indices]
            plotted_precision = precision_curve[indices]
        else:
            plotted_recall = recall_curve
            plotted_precision = precision_curve

        gd_prevalence = float(test_targets.mean())

        pr_figure, pr_axis = plt.subplots(
            figsize=(6.4, 5.4)
        )

        pr_axis.plot(
            plotted_recall,
            plotted_precision,
            color="#D95F02",
            linewidth=2.3,
            label=(
                f"Proposed model "
                f"(AUPRC = {auprc_value:.3f})"
            ),
        )

        pr_axis.axhline(
            gd_prevalence,
            color="#666666",
            linestyle=":",
            linewidth=1.4,
            label=(
                f"GD pixel prevalence = "
                f"{gd_prevalence:.3f}"
            ),
        )

        pr_axis.set_xlabel("Recall")
        pr_axis.set_ylabel("Precision")
        pr_axis.set_xlim(0.0, 1.01)
        pr_axis.set_ylim(0.0, 1.01)
        pr_axis.grid(alpha=0.22, linewidth=0.7)

        pr_axis.legend(
            loc="lower left",
            frameon=True,
            facecolor="white",
            edgecolor="#CCCCCC",
        )

        pr_figure.tight_layout()

        pr_png = (
            OUTPUT_DIR
            / "figure_proposed_model_pr_auprc.png"
        )

        pr_pdf = (
            OUTPUT_DIR
            / "figure_proposed_model_pr_auprc.pdf"
        )

        pr_figure.savefig(
            pr_png,
            dpi=600,
            bbox_inches="tight",
        )

        pr_figure.savefig(
            pr_pdf,
            bbox_inches="tight",
        )

        plt.show()
        plt.close(pr_figure)

        pd.DataFrame([{
            "model": "Proposed six-channel model",
            "test_images": 60,
            "evaluation": "no TTA",
            "pixel_level_auprc": auprc_value,
            "gd_pixel_prevalence": gd_prevalence,
            "number_of_pixels": int(len(test_targets)),
            "number_of_gd_pixels": int(test_targets.sum()),
        }]).to_csv(
            OUTPUT_DIR / "test_pixel_level_auprc.csv",
            index=False,
        )

        print("\nCreated:", pr_png)
        print("Created:", pr_pdf)
        print(f"Pixel-level AUPRC: {auprc_value:.6f}")

    except Exception as error:
        print("\nPR–AUPRC figure could not be generated:")
        print(error)

else:
    print(
        "\nPR–AUPRC figure was not generated because "
        "pixel-level test probabilities were not found."
    )
    print(
        "training_history.csv alone is not sufficient "
        "to calculate AUPRC."
    )
    print(
        "If model and test_loader are created in the session, "
        "run this same cell again."
    )

# ------------------------------------------------
# 8. SAVE SUMMARY
# ------------------------------------------------

summary = {
    "source_zip": str(result_zip_path),
    "history_csv": str(history_path),
    "best_validation_epoch": best_epoch,
    "best_logged_validation_dice": best_val_dice,
    "validation_selected_threshold": (
        VALIDATION_SELECTED_THRESHOLD
    ),
    "threshold_sensitivity_generated": (
        threshold_table is not None
    ),
    "pixel_level_auprc_generated": (
        auprc_value is not None
    ),
    "pixel_level_auprc": auprc_value,
    "evaluation_condition": "no TTA",
}

with open(
    OUTPUT_DIR / "figure_generation_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        indent=2,
        allow_nan=True,
    )

# ------------------------------------------------
# 9. CREATE DOWNLOADABLE ZIP
# ------------------------------------------------

download_base = WORK_DIR / "gd_seed42_paper_ready_figures"

download_zip = shutil.make_archive(
    str(download_base),
    "zip",
    root_dir=OUTPUT_DIR,
)

print("\n" + "=" * 66)
print("ALL AVAILABLE FIGURES COMPLETED")
print("=" * 66)
print("Download ZIP:", download_zip)

try:
    from IPython.display import FileLink, display

    display(FileLink(download_zip))

except Exception:
    pass